# Gravity Sewer Asset Health KPI Calculator

Calculates all 20 KPIs from the *Sample KPIs – Gravity Sewer* specification
using realistic mock pipe data. Pure Python standard library only (no numpy/pandas required).

In [1]:
import random, math, json, statistics, csv, io
random.seed(42)

## 1. Generate Mock Pipe Dataset
1 200 pipes spread across four drainage basins with realistic attribute distributions.

In [2]:
N = 1200

BASINS        = ['Basin_A', 'Basin_B', 'Basin_C', 'Basin_D']
MATERIALS     = ['VCP', 'PVC', 'RCP', 'DIP', 'CMP', 'HDPE']
MATERIAL_LIFE = {'VCP': 70, 'PVC': 100, 'RCP': 80, 'DIP': 90, 'CMP': 40, 'HDPE': 80}
MANNING_N     = {'VCP': 0.013, 'PVC': 0.010, 'RCP': 0.013, 'DIP': 0.012, 'CMP': 0.022, 'HDPE': 0.011}
INV_SOURCES   = ['survey', 'as-built', 'record_drawing', 'model-inferred', 'unknown']
INV_WEIGHTS   = {'survey': 1.0, 'as-built': 0.85, 'record_drawing': 0.70,
                 'model-inferred': 0.50, 'unknown': 0.0}
DIAMETERS     = [8, 10, 12, 15, 18, 24, 30, 36]
CURRENT_YEAR  = 2025
CURRENT_MODEL = 'v2024.1'

# Weighted random choice helper
def wchoice(population, weights):
    return random.choices(population, weights=weights, k=1)[0]

pipes = []
for i in range(1, N + 1):
    gis_len = round(random.uniform(50, 600), 1)

    # Model length = GIS ± small noise; ~6 % have > 10 % deviation
    if random.random() < 0.06:
        noise_frac = random.uniform(0.10, 0.25) * random.choice([-1, 1])
    else:
        noise_frac = random.gauss(0, 0.03)
    model_len = round(gis_len * (1 + noise_frac), 1)

    base_elev  = random.uniform(200, 400)
    drop       = random.uniform(0.1, 3.0)
    us_inv     = round(base_elev, 2)
    ds_inv     = round(base_elev - drop, 2)

    # ~1.2 % flat, ~0.8 % reversed slope
    rnd = random.random()
    if rnd < 0.008:
        ds_inv = round(us_inv + random.uniform(0.1, 1.0), 2)   # negative slope
    elif rnd < 0.020:
        ds_inv = us_inv                                          # flat

    slope = None if random.random() < 0.05 else round((us_inv - ds_inv) / gis_len, 6)

    mat    = wchoice(MATERIALS + [None], [28, 22, 18, 12, 8, 6, 6])
    diam   = wchoice(DIAMETERS + [None], [18, 15, 22, 16, 12, 8, 4, 3, 2])
    year   = None if random.random() < 0.07 else random.randint(1940, 2020)
    depth  = None if random.random() < 0.09 else round(random.uniform(4, 20), 1)
    us_mh  = None if random.random() < 0.008 else f'MH{random.randint(1,500):04d}'
    ds_mh  = None if random.random() < 0.008 else f'MH{random.randint(1,500):04d}'
    geom   = random.random() > 0.015
    basin  = wchoice(BASINS, [30, 25, 25, 20])
    sarea  = wchoice(['SA_North','SA_South','SA_East','SA_West'], [1,1,1,1])
    mdl_lnk = None if random.random() < 0.04 else f'LNK{i}'
    mdl_ver = CURRENT_MODEL if random.random() > 0.06 else 'v2022.3'
    us_src  = wchoice(INV_SOURCES, [25, 30, 20, 15, 10])
    ds_src  = wchoice(INV_SOURCES, [25, 30, 20, 15, 10])
    rim_src = wchoice(INV_SOURCES, [30, 28, 18, 14, 10])

    pipes.append({
        'pipe_id': f'P{i:05d}', 'basin': basin, 'service_area': sarea,
        'has_geometry': geom, 'diameter_in': diam, 'gis_length_ft': gis_len,
        'model_length_ft': model_len, 'material': mat, 'install_year': year,
        'avg_depth_ft': depth, 'us_mh': us_mh, 'ds_mh': ds_mh,
        'us_invert_elev': us_inv, 'ds_invert_elev': ds_inv, 'slope': slope,
        'model_link_id': mdl_lnk, 'model_version': mdl_ver,
        'us_invert_source': us_src, 'ds_invert_source': ds_src,
        'rim_elev_source': rim_src,
    })

print(f'Generated {N} pipes across {len(BASINS)} basins')
print('Sample record:', {k: v for k, v in list(pipes[0].items())[:6]})

Generated 1200 pipes across 4 basins
Sample record: {'pipe_id': 'P00001', 'basin': 'Basin_C', 'service_area': 'SA_West', 'has_geometry': True, 'diameter_in': 8, 'gis_length_ft': 401.7}


## 2. KPI Calculations

In [3]:
results = {}

def pct(num, den):
    return round(num / den * 100, 2) if den > 0 else 0.0

def count(field, predicate=None, dataset=None):
    ds = dataset if dataset is not None else pipes
    if predicate is None:
        return sum(1 for p in ds if p.get(field) is not None)
    return sum(1 for p in ds if predicate(p))

In [4]:
# GS_AHC_001 – Pipe Inventory Completeness
valid_inv = sum(1 for p in pipes
                if p['pipe_id'] and p['has_geometry'] and p['basin'] and p['service_area'])
kpi_001 = pct(valid_inv, N)
results['GS_AHC_001'] = kpi_001
print(f'GS_AHC_001  Pipe Inventory Completeness          : {kpi_001:.2f}%')

GS_AHC_001  Pipe Inventory Completeness          : 98.00%


In [5]:
# GS_AHC_002 – Critical Design Field Completeness
def all_critical(p):
    return all(p.get(f) is not None for f in
               ['diameter_in','gis_length_ft','slope','material',
                'install_year','avg_depth_ft','us_mh','ds_mh'])

kpi_002 = pct(sum(1 for p in pipes if all_critical(p)), N)
results['GS_AHC_002'] = kpi_002
print(f'GS_AHC_002  Critical Design Field Completeness   : {kpi_002:.2f}%')

GS_AHC_002  Critical Design Field Completeness   : 75.50%


In [6]:
# GS_AHC_003 – Diameter Completeness
kpi_003 = pct(count('diameter_in'), N)
results['GS_AHC_003'] = kpi_003
print(f'GS_AHC_003  Diameter Completeness                : {kpi_003:.2f}%')

GS_AHC_003  Diameter Completeness                : 98.33%


In [7]:
# GS_AHC_004 – Slope Completeness
kpi_004 = pct(count('slope'), N)
results['GS_AHC_004'] = kpi_004
print(f'GS_AHC_004  Slope Completeness                   : {kpi_004:.2f}%')

GS_AHC_004  Slope Completeness                   : 94.67%


In [8]:
# GS_AHC_005 – Length Consistency  (mean % difference across all pipes)
diffs = [abs(p['gis_length_ft'] - p['model_length_ft']) / p['gis_length_ft'] * 100
         for p in pipes]
kpi_005 = round(statistics.mean(diffs), 2)
results['GS_AHC_005'] = kpi_005
print(f'GS_AHC_005  Length Consistency (mean % diff)     : {kpi_005:.2f}%')

GS_AHC_005  Length Consistency (mean % diff)     : 3.30%


In [9]:
# GS_AHC_006 – Material Completeness
kpi_006 = pct(count('material'), N)
results['GS_AHC_006'] = kpi_006
print(f'GS_AHC_006  Material Completeness                : {kpi_006:.2f}%')

GS_AHC_006  Material Completeness                : 95.58%


In [10]:
# GS_AHC_007 – Age Completeness
kpi_007 = pct(count('install_year'), N)
results['GS_AHC_007'] = kpi_007
print(f'GS_AHC_007  Age Completeness                     : {kpi_007:.2f}%')

GS_AHC_007  Age Completeness                     : 94.33%


In [11]:
# GS_AHC_008 – Depth Completeness
kpi_008 = pct(count('avg_depth_ft'), N)
results['GS_AHC_008'] = kpi_008
print(f'GS_AHC_008  Depth Completeness                   : {kpi_008:.2f}%')

GS_AHC_008  Depth Completeness                   : 91.08%


In [12]:
# GS_AHC_009 – Manhole Connectivity Completeness
kpi_009 = pct(sum(1 for p in pipes if p['us_mh'] and p['ds_mh']), N)
results['GS_AHC_009'] = kpi_009
print(f'GS_AHC_009  Manhole Connectivity Completeness    : {kpi_009:.2f}%')

GS_AHC_009  Manhole Connectivity Completeness    : 98.58%


In [13]:
# GS_AHC_010 – Orphan Pipe Rate
orphan = sum(1 for p in pipes if not p['us_mh'] or not p['ds_mh'])
kpi_010 = pct(orphan, N)
results['GS_AHC_010'] = kpi_010
print(f'GS_AHC_010  Orphan Pipe Rate                     : {kpi_010:.2f}%')

GS_AHC_010  Orphan Pipe Rate                     : 1.42%


In [14]:
# GS_AHC_011 – GIS-to-Model Linkage Completeness
kpi_011 = pct(count('model_link_id'), N)
results['GS_AHC_011'] = kpi_011
print(f'GS_AHC_011  GIS-to-Model Linkage Completeness    : {kpi_011:.2f}%')

GS_AHC_011  GIS-to-Model Linkage Completeness    : 96.25%


In [15]:
# GS_AHC_012 – Model Link Freshness
linked = [p for p in pipes if p['model_link_id']]
fresh  = sum(1 for p in linked if p['model_version'] == CURRENT_MODEL)
kpi_012 = pct(fresh, len(linked))
results['GS_AHC_012'] = kpi_012
print(f'GS_AHC_012  Model Link Freshness                 : {kpi_012:.2f}%')

GS_AHC_012  Model Link Freshness                 : 94.20%


In [16]:
# GS_AHC_013 – Invert Elevation Confidence  (0–100 scale)
pipe_invert_scores = [
    (INV_WEIGHTS[p['us_invert_source']] + INV_WEIGHTS[p['ds_invert_source']]) / 2
    for p in pipes
]
kpi_013 = round(statistics.mean(pipe_invert_scores) * 100, 1)
results['GS_AHC_013'] = kpi_013
print(f'GS_AHC_013  Invert Elevation Confidence          : {kpi_013:.1f}/100')

GS_AHC_013  Invert Elevation Confidence          : 71.7/100


In [17]:
# GS_AHC_014 – Rim Elevation Confidence  (0–100 scale)
rim_scores = [INV_WEIGHTS[p['rim_elev_source']] for p in pipes]
kpi_014 = round(statistics.mean(rim_scores) * 100, 1)
results['GS_AHC_014'] = kpi_014
print(f'GS_AHC_014  Rim Elevation Confidence             : {kpi_014:.1f}/100')

GS_AHC_014  Rim Elevation Confidence             : 73.0/100


In [18]:
# GS_AHC_015 – Pipe Slope Reasonableness Pass Rate
# Reasonable gravity sewer slope: 0.001 ft/ft  ≤  slope  ≤  0.10 ft/ft
has_slope = [p for p in pipes if p['slope'] is not None]
reasonable = sum(1 for p in has_slope if 0.001 <= p['slope'] <= 0.10)
kpi_015 = pct(reasonable, len(has_slope))
results['GS_AHC_015'] = kpi_015
print(f'GS_AHC_015  Pipe Slope Reasonableness Pass Rate  : {kpi_015:.2f}%')

GS_AHC_015  Pipe Slope Reasonableness Pass Rate  : 91.11%


In [19]:
# GS_AHC_016 – Negative or Flat Slope Rate
neg_or_flat = sum(1 for p in has_slope if p['slope'] <= 0.0001)
kpi_016 = pct(neg_or_flat, len(has_slope))
results['GS_AHC_016'] = kpi_016
print(f'GS_AHC_016  Negative or Flat Slope Rate          : {kpi_016:.2f}%')

GS_AHC_016  Negative or Flat Slope Rate          : 2.73%


In [20]:
# GS_AHC_017 – Design Capacity Estimate Completeness
# Requires: diameter, slope, material, length
cap_ready = sum(1 for p in pipes if all(
    p.get(f) is not None for f in ['diameter_in','slope','material','gis_length_ft']
))
kpi_017 = pct(cap_ready, N)
results['GS_AHC_017'] = kpi_017
print(f'GS_AHC_017  Design Capacity Estimate Completeness: {kpi_017:.2f}%')

GS_AHC_017  Design Capacity Estimate Completeness: 88.92%


In [21]:
# GS_AHC_018 – Full-Pipe Design Capacity  (Manning's Equation)
# Q = (1.486 / n) * A * R^(2/3) * S^(1/2)   [US customary units]
def manning_q(diam_in, slope, material):
    if diam_in is None or slope is None or material is None or slope <= 0:
        return None
    n = MANNING_N.get(material, 0.013)
    d = diam_in / 12               # ft
    A = math.pi * d**2 / 4        # ft²
    R = d / 4                      # hydraulic radius for full circle
    return round((1.486 / n) * A * R**(2/3) * slope**0.5, 4)

capacities = [manning_q(p['diameter_in'], p['slope'], p['material']) for p in pipes]
cap_valid  = [c for c in capacities if c is not None]

sorted_caps = sorted(cap_valid)
mid = len(sorted_caps) // 2
kpi_018 = round(
    (sorted_caps[mid - 1] + sorted_caps[mid]) / 2 if len(sorted_caps) % 2 == 0
    else sorted_caps[mid], 3
)
results['GS_AHC_018'] = kpi_018
print(f'GS_AHC_018  Median Full-Pipe Design Capacity     : {kpi_018:.3f} cfs')
print(f'            Pipes with capacity estimate         : {len(cap_valid)}')

GS_AHC_018  Median Full-Pipe Design Capacity     : 3.642 cfs
            Pipes with capacity estimate         : 1038


In [22]:
# GS_AHC_019 – Asset Age Exceedance Rate
with_age = [p for p in pipes if p['install_year'] is not None and p['material'] is not None]
exceeded = sum(1 for p in with_age
               if (CURRENT_YEAR - p['install_year']) > MATERIAL_LIFE.get(p['material'], 75))
kpi_019 = pct(exceeded, len(with_age))
results['GS_AHC_019'] = kpi_019
print(f'GS_AHC_019  Asset Age Exceedance Rate            : {kpi_019:.2f}%')

GS_AHC_019  Asset Age Exceedance Rate            : 12.40%


In [23]:
# GS_AHC_020 – Basin Asset Backbone Completeness Score  (0–100)
# Weighted composite of five component KPIs:
#   25% Inventory Completeness (001)
#   20% Critical Field Completeness (002)
#   15% Manhole Connectivity (009)
#   20% Invert Elevation Confidence (013, already 0-100)
#   10% GIS-to-Model Linkage (011)
#   10% Model Link Freshness (012)
weights = {
    'GS_AHC_001': 0.25,
    'GS_AHC_002': 0.20,
    'GS_AHC_009': 0.15,
    'GS_AHC_013': 0.20,
    'GS_AHC_011': 0.10,
    'GS_AHC_012': 0.10,
}
kpi_020 = round(sum(results[k] * v for k, v in weights.items()), 1)
results['GS_AHC_020'] = kpi_020
print(f'GS_AHC_020  Basin Asset Backbone Score           : {kpi_020:.1f}/100')

GS_AHC_020  Basin Asset Backbone Score           : 87.8/100


## 3. Classify and Summarise All KPIs

In [24]:
KPI_META = [
    ('GS_AHC_001','Pipe Inventory Completeness',          '%',      'Good ≥ 98; Watch 95–98; Action < 95'),
    ('GS_AHC_002','Critical Design Field Completeness',   '%',      'Good ≥ 95; Watch 90–95; Action < 90'),
    ('GS_AHC_003','Diameter Completeness',                '%',      'Good ≥ 99; Action < 98'),
    ('GS_AHC_004','Slope Completeness',                   '%',      'Good ≥ 95; Action < 90'),
    ('GS_AHC_005','Length Consistency (mean % diff)',     '% diff', 'Good ≤ 5; Watch 5–10; Action > 10'),
    ('GS_AHC_006','Material Completeness',                '%',      'Good ≥ 95; Watch 90–95; Action < 90'),
    ('GS_AHC_007','Age Completeness',                     '%',      'Good ≥ 95; Action < 90'),
    ('GS_AHC_008','Depth Completeness',                   '%',      'Good ≥ 90; Action < 85'),
    ('GS_AHC_009','Manhole Connectivity Completeness',    '%',      'Good ≥ 99; Action < 98'),
    ('GS_AHC_010','Orphan Pipe Rate',                     '%',      'Good < 1; Action ≥ 2'),
    ('GS_AHC_011','GIS-to-Model Linkage Completeness',   '%',      'Good ≥ 95; Watch 85–95; Action < 85'),
    ('GS_AHC_012','Model Link Freshness',                 '%',      'Good ≥ 95; Watch 85–95; Action < 85'),
    ('GS_AHC_013','Invert Elevation Confidence',          '0–100',  'Good ≥ 85; Watch 65–85; Action < 65'),
    ('GS_AHC_014','Rim Elevation Confidence',             '0–100',  'Good ≥ 85; Action < 65'),
    ('GS_AHC_015','Pipe Slope Reasonableness Pass Rate',  '%',      'Good ≥ 95; Action < 90'),
    ('GS_AHC_016','Negative or Flat Slope Rate',          '%',      'Good < 1; Action ≥ 2'),
    ('GS_AHC_017','Design Capacity Estimate Completeness','%',      'Good ≥ 95; Action < 90'),
    ('GS_AHC_018','Median Full-Pipe Design Capacity',     'cfs',    'Compare by diameter/slope/design std'),
    ('GS_AHC_019','Asset Age Exceedance Rate',            '%',      'Watch > 20; Action > 35'),
    ('GS_AHC_020','Basin Asset Backbone Completeness',    '0–100',  'Certified ≥ 90; Provisional 75–90; Not Certified < 75'),
]

def classify(kid, v):
    if kid == 'GS_AHC_005':  return 'Good' if v <= 5 else ('Watch' if v <= 10 else 'Action')
    if kid == 'GS_AHC_010':  return 'Good' if v < 1  else ('Watch' if v < 2  else 'Action')
    if kid == 'GS_AHC_016':  return 'Good' if v < 1  else ('Watch' if v < 2  else 'Action')
    if kid == 'GS_AHC_019':  return 'Good' if v <= 20 else ('Watch' if v <= 35 else 'Action')
    if kid == 'GS_AHC_020':  return 'Certified' if v >= 90 else ('Provisional' if v >= 75 else 'Not Certified')
    if kid == 'GS_AHC_018':  return 'Info'
    if kid in ('GS_AHC_013','GS_AHC_014'): return 'Good' if v >= 85 else ('Watch' if v >= 65 else 'Action')
    if kid in ('GS_AHC_011','GS_AHC_012'): return 'Good' if v >= 95 else ('Watch' if v >= 85 else 'Action')
    thresholds = {
        'GS_AHC_001':(98,95),'GS_AHC_002':(95,90),'GS_AHC_003':(99,98),
        'GS_AHC_004':(95,90),'GS_AHC_006':(95,90),'GS_AHC_007':(95,90),
        'GS_AHC_008':(90,85),'GS_AHC_009':(99,98),'GS_AHC_015':(95,90),
        'GS_AHC_017':(95,90),
    }
    hi, lo = thresholds.get(kid, (95, 90))
    return 'Good' if v >= hi else ('Watch' if v >= lo else 'Action')

summary = []
for kid, name, units, bands in KPI_META:
    v   = results[kid]
    sts = classify(kid, v)
    summary.append({'KPI_ID': kid, 'KPI_Name': name, 'Value': v,
                    'Units': units, 'Status': sts, 'Bands': bands})

# Print table
header = f"{'KPI_ID':<14} {'KPI_Name':<42} {'Value':>8} {'Units':<9} {'Status'}"
print(header)
print('-' * len(header))
for r in summary:
    print(f"{r['KPI_ID']:<14} {r['KPI_Name']:<42} {r['Value']:>8.2f} {r['Units']:<9} {r['Status']}")

KPI_ID         KPI_Name                                      Value Units     Status
-----------------------------------------------------------------------------------
GS_AHC_001     Pipe Inventory Completeness                   98.00 %         Good
GS_AHC_002     Critical Design Field Completeness            75.50 %         Action
GS_AHC_003     Diameter Completeness                         98.33 %         Watch
GS_AHC_004     Slope Completeness                            94.67 %         Watch
GS_AHC_005     Length Consistency (mean % diff)               3.30 % diff    Good
GS_AHC_006     Material Completeness                         95.58 %         Good
GS_AHC_007     Age Completeness                              94.33 %         Watch
GS_AHC_008     Depth Completeness                            91.08 %         Good
GS_AHC_009     Manhole Connectivity Completeness             98.58 %         Watch
GS_AHC_010     Orphan Pipe Rate                               1.42 %         Watch
GS_AH

## 4. Basin-Level Breakdown

In [25]:
basin_stats = []
for basin in BASINS:
    b = [p for p in pipes if p['basin'] == basin]
    n = len(b)
    bs = [
        basin, n,
        pct(sum(1 for p in b if p['pipe_id'] and p['has_geometry'] and p['basin'] and p['service_area']), n),
        pct(sum(1 for p in b if p['diameter_in'] is not None), n),
        pct(sum(1 for p in b if p['material'] is not None), n),
        pct(sum(1 for p in b if p['us_mh'] and p['ds_mh']), n),
        pct(sum(1 for p in b if p['slope'] is not None and p['slope'] <= 0.0001),
            max(1, sum(1 for p in b if p['slope'] is not None))),
        pct(sum(1 for p in b if p['model_link_id'] is not None), n),
    ]
    basin_stats.append(bs)

cols = ['Basin','Pipes','InvComp%','DiamComp%','MatComp%','MHConn%','NegSlope%','ModelLink%']
print('  '.join(f'{c:<12}' for c in cols))
print('-' * 110)
for row in basin_stats:
    print('  '.join(f'{str(v):<12}' for v in row))

basin_records = [dict(zip(cols, row)) for row in basin_stats]

Basin         Pipes         InvComp%      DiamComp%     MatComp%      MHConn%       NegSlope%     ModelLink%  
--------------------------------------------------------------------------------------------------------------
Basin_A       381           97.38         97.64         96.85         97.9          2.76          95.01       
Basin_B       299           97.99         98.33         93.98         98.33         2.52          96.32       
Basin_C       298           98.32         98.66         95.64         99.66         2.8           98.32       
Basin_D       222           98.65         99.1          95.5          98.65         2.86          95.5        


## 5. Export to JSON for Dashboard

In [26]:
with open('kpi_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

with open('basin_kpi.json', 'w') as f:
    json.dump(basin_records, f, indent=2)

print('Exported kpi_results.json and basin_kpi.json')

Exported kpi_results.json and basin_kpi.json
